In [ ]:
import sys 
sys.path.append('/home/lumargot/trachoma/src/py')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # put -1 to not use any
from torch import Tensor, nn
import torchvision

In [ ]:
TORCHDYNAMO_VERBOSE=1
TORCH_LOGS="+dynamo"

In [ ]:
from loaders.tt_dataset import BBXImageTestTransform, BBXImageEvalTransform
import SimpleITK as sitk
from nets.segmentation import FasterTTRCNN, MobileFasterTTRCNN
import torch
import numpy as np
from nets.segmentation import FasterTTRCNN
import matplotlib.pyplot as plt 
from matplotlib.patches import Rectangle

import torch
import torchvision.models as models
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

In [ ]:

ckpt = '/CMF/data/lumargot/trachoma/output/backtoold/5fold_v4/fold2/epoch=20-val_loss=1.83.ckpt'

In [ ]:
class TTUSegTorch(nn.Module):
    def __init__(self, faster, max_detections):
        super(TTUSegTorch, self).__init__()
        self.faster = faster
        self.max_detections = max_detections
        self.faster.eval()

    def forward(self, x):
        x = self.faster(x)
        num_detections = min(len(x[0]['boxes']), self.max_detections)

        boxes = torch.zeros((self.max_detections, 4), dtype=torch.float32)
        scores = torch.zeros(self.max_detections, dtype=torch.float32)
        labels = torch.zeros(self.max_detections, dtype=torch.float32)

        if num_detections > 0:
            boxes = x[0]['boxes'][:num_detections,:]
            scores = x[0]['scores'][:num_detections]
            labels = x[0]['labels'][:num_detections]
            
        return boxes, labels.float(), scores 
    


In [ ]:
model = FasterTTRCNN(out_features=4, class_weights = torch.ones(4))

state_dict = torch.load(ckpt, weights_only=False, map_location='cpu') # or 'cuda' if loading to GPU

# Load the state dictionary into the model
model.load_state_dict(state_dict['state_dict'])

# Set the model to evaluation mode if you are using it for inference
model.eval()

In [ ]:
model_2 = TTUSegTorch(model.model, max_detections=25)
model_2.eval()

In [ ]:

# model = models.mobilenetv2.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
sample_inputs = (torch.randn(1, 3, 384, 768), )

et_program = to_edge_transform_and_lower(
    torch.export.export(model_2, sample_inputs),
    partitioner=[XnnpackPartitioner()]
).to_executorch()

with open("model-example.pte", "wb") as f:
    f.write(et_program.buffer)